### Algebraic Helmholtz Inversion

Implements AHI and applies to clean and noisy data.

In [ ]:
# Algebraic Helmholtz Inversion

import os, sys

sys.path.append("..")

import numpy as np
import matplotlib.pyplot as plt
from scipy.io import loadmat
from hetsi import diff as hsd

In [ ]:
# Load data

fpath = "_PATH_TO_DATA_" # Path to data folder

def loadData(path):
    fname = "four_target_phantom.mat"

    if not path.endswith("/"):
        path += "/"

    fpath = path + fname

    disp = loadmat(fpath)["u_ft"]
    disp = np.transpose(disp, (4, 1, 0, 2, 3)).astype(np.complex128)

    return disp

# Load
disp = loadData(fpath)

# General dataset parameters
y_spacing = np.array([1, 0.001, 0.001, 0.001, 1])
frequencies = np.array([50, 60, 70, 80, 90, 100])


In [ ]:

def ahi_inversion(data, spacing, rho_estimate = 1000):
    """Simple 3D Algebraic Helmholtz Inversion (AHI)."""

    # Laplacian
    dx = hsd.fd(data, spacing, axes = (1,2,3))
    dx2 = hsd.fd(dx, spacing, axes = (1,2,3))
    lapl = hsd.np.trace(dx2, axis1=-2, axis2=-1)

    # 2nd Time Derivative
    dt2 = -((2 * np.pi * frequencies[..., *[None]*4])**2) * data

    # Invert
    mu_estimate = rho_estimate * dt2 / lapl
    mu_estimate = np.mean(mu_estimate, axis = (0,)) # Mean of frequencies

    return mu_estimate

mu = ahi_inversion(disp, y_spacing)
mu = np.transpose(mu, (1,0,2,3))

np.save("ahi_output.npy", mu)

In [ ]:
# Output

fig = plt.figure()
ax = plt.imshow(np.abs(mu[:,:,5,2]), vmax = 15000)
plt.colorbar()
plt.title("AHI")

# fig.savefig("ahim.png",
#             transparent=True,
#             dpi = 600,
#             format = "png")

### Noisy data

In [ ]:
# Load data

noisy_data = np.load("...") # Path to noisy dataset generated from "data/gen_noisy_data.ipynb"

In [ ]:
# Inversion

data_split = [noisy_data[i, ...] for i in range(noisy_data.shape[0])]
noisy_estimate = []
spacing = np.array([1e3, 1,1,1, 1e3]) * 1e-3
frequencies = np.array([50, 60, 70, 80, 90, 100])

for d in data_split:
    noisy_estimate.append(ahi_inversion(d, spacing))

noisy_estimate = np.stack(noisy_estimate, axis = 0)

In [ ]:
# Print results

fig, axs = plt.subplots(1,5)

for i in range(noisy_estimate.shape[0]):
    im = axs[i].imshow(np.abs(noisy_estimate[i, :, :, 5, 2]), vmax = 15000)

In [ ]:
# Save inversion results

np.save("noisy_inverse.npy", noisy_estimate)